# 01 - VietNews Dataset EDA And Understanding

This notebook is for **understanding the VietNews dataset before it is used by TF-IDF, TextRank, and PhoBERT notebooks**. It focuses on data exploration, not model training, not model benchmarking, and not official artifact export.

Run the notebook sequentially from the top so variables such as `manifest`, `processed_path`, and `df` stay consistent.

Questions answered here:

- Which dataset splits are available, and how many samples are in each split?
- What fields exist in raw records and processed records?
- What are the source text and gold summary for this summarization task?
- Are required fields missing or empty?
- How long are articles and reference summaries?
- Is the compression ratio reasonable for text summarization?

Project convention: `article` is the input document, and `reference_summary` is the gold summary used for evaluation. In the raw dataset, this gold summary is named `abstract`.


## 1. Task Context And Dataset Role

In this project, each example is a Vietnamese news article paired with a reference summary. TF-IDF, TextRank, and PhoBERT will receive `article` as the input document and produce a predicted summary. That prediction is later compared with `reference_summary` using metrics such as ROUGE, compression ratio, and repetition rate.

Before running any model, this notebook clarifies three things:

- Whether the data has the right structure for the `article -> summary` task.
- Whether article and summary lengths are reasonable for summarization.
- Whether the benchmark split is clean enough for fair method comparison.


## 2. Minimal Setup

This cell only configures paths and imports basic EDA libraries. The notebook does not import summarization engines, inspect git state, or export official artifacts.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd().resolve()
if (ROOT / "backend").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "backend").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Repository root not found. Open the notebook from the repo root or notebooks/.")

TARGET_SPLIT = "validation"  # có thể đổi thành "train" hoặc "test" để EDA split khác
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "vietnews"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "vietnews"
MANIFEST_PATH = PROCESSED_DIR / "dataset_manifest.json"

pd.set_option("display.max_colwidth", 180)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("TARGET_SPLIT:", TARGET_SPLIT)


## 3. Dataset Size And Splits

`dataset_manifest.json` records the preprocessing protocol, split sizes, and rows dropped during dataset preparation. This is the first checkpoint for understanding what data is available before benchmarking.


In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

with MANIFEST_PATH.open(encoding="utf-8") as f:
    manifest = json.load(f)

print("protocol_version:", manifest.get("protocol_version"))

split_rows = []
for split_name, split_info in (manifest.get("splits") or {}).items():
    split_rows.append(
        {
            "split": split_name,
            "raw_rows": split_info.get("raw_rows"),
            "written_rows": split_info.get("written_rows"),
            "dropped_empty_article": split_info.get("dropped_empty_article"),
            "dropped_empty_reference_summary": split_info.get("dropped_empty_reference_summary"),
            "json_decode_errors": split_info.get("json_decode_errors"),
        }
    )

split_order = {"train": 0, "validation": 1, "test": 2}
split_df = pd.DataFrame(split_rows)
split_df["split_order"] = split_df["split"].map(split_order).fillna(99)
split_df = split_df.sort_values(["split_order", "split"]).drop(columns="split_order").reset_index(drop=True)
split_df["row_share"] = split_df["written_rows"] / split_df["written_rows"].sum()
display(split_df)

global_stats = manifest.get("global_stats") or {}
if global_stats:
    display(pd.Series(global_stats, name="global_stats"))


## 4. Raw Schema Versus Processed Schema

Raw VietNews uses `abstract` as the reference summary. After preprocessing, the project renames this field to `reference_summary` so model and evaluation notebooks can use a clearer field name.


In [ ]:
raw_path = RAW_DIR / f"{TARGET_SPLIT}.jsonl"
processed_path = PROCESSED_DIR / f"{TARGET_SPLIT}.jsonl"

if not processed_path.exists():
    raise FileNotFoundError(f"Processed split not found: {processed_path}")

raw_sample = None
if raw_path.exists():
    with raw_path.open(encoding="utf-8") as f:
        raw_sample = json.loads(f.readline())

with processed_path.open(encoding="utf-8") as f:
    processed_sample = json.loads(f.readline())

schema_rows = []
if raw_sample is not None:
    schema_rows.append(
        {
            "view": "raw",
            "columns": ", ".join(raw_sample.keys()),
            "source_text": "article",
            "target_summary": "abstract",
        }
    )

schema_rows.append(
    {
        "view": "processed",
        "columns": ", ".join(processed_sample.keys()),
        "source_text": "article",
        "target_summary": "reference_summary",
    }
)

display(pd.DataFrame(schema_rows))

sample_rows = [
    {"field": "guid", "value": processed_sample.get("guid")},
    {"field": "title", "value": processed_sample.get("title")},
    {"field": "article_preview", "value": str(processed_sample.get("article", ""))[:500]},
    {"field": "reference_summary", "value": processed_sample.get("reference_summary")},
    {"field": "meta", "value": processed_sample.get("meta")},
]

if raw_sample is not None:
    sample_rows.insert(3, {"field": "raw_abstract", "value": raw_sample.get("abstract")})

pd.DataFrame(sample_rows)


## 5. Load The Processed Split

Downstream benchmark notebooks should use processed data so every method shares the same preprocessing. This notebook defaults to `validation`, which is the main benchmark split currently used in the project.


In [ ]:
rows = []
with processed_path.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
if df.empty:
    raise RuntimeError(f"Split {TARGET_SPLIT!r} is empty.")

print("Loaded split:", TARGET_SPLIT)
print("Rows:", len(df))

preview_df = df[["guid", "title", "article", "reference_summary", "meta"]].head(3).copy()
preview_df["article"] = preview_df["article"].str.slice(0, 260) + " ..."
preview_df["reference_summary"] = preview_df["reference_summary"].str.slice(0, 260)
preview_df


## 6. Schema, Missing Values, And Empty Text

This section checks the practical conditions required before sending data into models: required columns exist, `article` and `reference_summary` are not empty, `guid` is not duplicated, and `meta` contains the length fields needed for EDA.

`quick_checks` includes duplicate `guid` rows, excluding the first occurrence, and rows where `meta` is not a dictionary.


In [ ]:
required_cols = ["guid", "title", "article", "reference_summary", "meta"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

schema_rows = []
for col in required_cols:
    values = df[col]
    empty_count = 0
    if col != "meta":
        empty_count = int((values.fillna("").astype(str).str.strip() == "").sum())
    schema_rows.append(
        {
            "column": col,
            "dtype": str(values.dtype),
            "null_count": int(values.isna().sum()),
            "empty_count": empty_count,
            "empty_ratio": empty_count / len(df),
        }
    )

schema_check_df = pd.DataFrame(schema_rows)
display(schema_check_df)

quick_checks = pd.Series(
    {
        "duplicate_guid_rows_excluding_first": int(df["guid"].duplicated().sum()),
        "non_dict_meta_rows": int((~df["meta"].map(lambda m: isinstance(m, dict))).sum()),
    },
    name="quick_checks",
)
display(quick_checks)

required_meta_keys = [
    "protocol_version",
    "article_char_len",
    "reference_summary_char_len",
    "article_sentence_count",
    "reference_summary_sentence_count",
]

meta_key_df = pd.DataFrame(
    [
        {
            "meta_key": key,
            "present_ratio": float(df["meta"].map(lambda meta: isinstance(meta, dict) and key in meta).mean()),
        }
        for key in required_meta_keys
    ]
)
meta_key_df


## 7. Length And Compression Ratio

Character length and sentence count come from `meta`, which was created during preprocessing. Token count here is only a whitespace-based EDA proxy; it is not a replacement for the PhoBERT tokenizer.

The `describe` table summarizes length distributions. `sanity_checks` flags unusual cases such as summaries longer than articles, compression ratio greater than 1, or non-positive article sentence counts.


In [ ]:
meta_series = df["meta"]

df["article_char_len"] = meta_series.map(lambda meta: int((meta or {}).get("article_char_len") or 0))
df["reference_char_len"] = meta_series.map(lambda meta: int((meta or {}).get("reference_summary_char_len") or 0))
df["article_sentence_count"] = meta_series.map(lambda meta: int((meta or {}).get("article_sentence_count") or 0))
df["reference_sentence_count"] = meta_series.map(lambda meta: int((meta or {}).get("reference_summary_sentence_count") or 0))

df["article_token_count"] = df["article"].fillna("").astype(str).str.split().str.len()
df["reference_token_count"] = df["reference_summary"].fillna("").astype(str).str.split().str.len()
df["compression_ratio_chars"] = df["reference_char_len"] / df["article_char_len"].clip(lower=1)
df["compression_ratio_sentences"] = df["reference_sentence_count"] / df["article_sentence_count"].clip(lower=1)

summary_metrics = [
    "article_char_len",
    "reference_char_len",
    "article_token_count",
    "reference_token_count",
    "article_sentence_count",
    "reference_sentence_count",
    "compression_ratio_chars",
    "compression_ratio_sentences",
]

stats_df = df[summary_metrics].describe(percentiles=[0.5, 0.9, 0.95]).T
stats_df = stats_df[["count", "mean", "50%", "90%", "95%", "min", "max"]].rename(
    columns={"50%": "p50", "90%": "p90", "95%": "p95"}
)
display(stats_df)

sanity_checks = pd.Series(
    {
        "summary_chars_gt_article_chars": int((df["reference_char_len"] > df["article_char_len"]).sum()),
        "compression_ratio_chars_gt_1": int((df["compression_ratio_chars"] > 1).sum()),
        "article_sentence_count_non_positive": int((df["article_sentence_count"] <= 0).sum()),
    },
    name="sanity_checks",
)
sanity_checks


## 8. Length Distributions

Histograms make it easier to see whether most articles and summaries are short or long. Values are clipped at p99 so extreme outliers do not dominate the x-axis.


In [ ]:
if plt is None:
    print("matplotlib is not available in this environment, so histograms are skipped.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    plot_cols = [
        "article_char_len",
        "reference_char_len",
        "compression_ratio_chars",
        "article_sentence_count",
        "reference_sentence_count",
        "compression_ratio_sentences",
    ]

    for ax, col in zip(axes.flat, plot_cols):
        upper = df[col].quantile(0.99)
        df[col].clip(upper=upper).hist(ax=ax, bins=50)
        ax.set_title(f"{col} (<= p99)")
        ax.set_xlabel(col)
        ax.set_ylabel("count")

    plt.tight_layout()
    plt.show()


## 9. Manual Sample Inspection

EDA should include reading real examples, not only aggregate tables. This section checks whether the relationship between the source article and reference summary is reasonable.


In [ ]:
SEED = 42
sample_cols = [
    "guid",
    "title",
    "article_char_len",
    "article_sentence_count",
    "reference_char_len",
    "reference_sentence_count",
    "compression_ratio_chars",
    "compression_ratio_sentences",
]

sample_df = df.sample(n=min(5, len(df)), random_state=SEED).copy()
display(sample_df[sample_cols].sort_values("article_char_len", ascending=False))

for sample_no, (idx, row) in enumerate(sample_df.head(2).iterrows(), start=1):
    print("=" * 100)
    print(f"Sample {sample_no} | guid={row['guid']}")
    print("Title:", row["title"])
    print("-" * 100)
    print("Article preview:")
    print(str(row["article"])[:900], "...")
    print("-" * 100)
    print("Reference summary:")
    print(str(row["reference_summary"]))
    print("-" * 100)
    print(
        "article_chars={article_chars}, summary_chars={summary_chars}, compression_chars={compression:.4f}".format(
            article_chars=row["article_char_len"],
            summary_chars=row["reference_char_len"],
            compression=row["compression_ratio_chars"],
        )
    )


## 10. EDA Summary For Later Notebooks

The final cell collects the most important values into a short table. These values can be reused in the dataset description section of the project report.


In [ ]:
eda_summary = {
    "protocol_version": manifest.get("protocol_version"),
    "target_split": TARGET_SPLIT,
    "rows": int(len(df)),
    "article_empty_count": int((df["article"].fillna("").astype(str).str.strip() == "").sum()),
    "reference_summary_empty_count": int((df["reference_summary"].fillna("").astype(str).str.strip() == "").sum()),
    "split_total_rows": int(split_df["written_rows"].sum()),
    "article_char_p50": float(df["article_char_len"].quantile(0.50)),
    "reference_char_p50": float(df["reference_char_len"].quantile(0.50)),
    "article_token_p50": float(df["article_token_count"].quantile(0.50)),
    "article_token_p90": float(df["article_token_count"].quantile(0.90)),
    "reference_token_p50": float(df["reference_token_count"].quantile(0.50)),
    "reference_token_p90": float(df["reference_token_count"].quantile(0.90)),
    "compression_ratio_chars_mean": float(df["compression_ratio_chars"].mean()),
    "compression_ratio_chars_p90": float(df["compression_ratio_chars"].quantile(0.90)),
}

pd.Series(eda_summary, name="eda_summary")


### Notes For Downstream Experiments

- Notebook 02, 03, and 04 should use `data/processed/vietnews/{TARGET_SPLIT}.jsonl` to keep preprocessing consistent.
- `article` is the source document passed into the summarizer.
- `reference_summary` is the gold summary used for ROUGE, compression, and error analysis.
- Token counts in this notebook are only EDA proxies. PhoBERT needs its own tokenizer in the model pipeline.
- This notebook does not export official QA artifacts. If official artifacts are needed, keep that responsibility in scripts such as `scripts/generate_official_validation_artifacts.py`.
